# Adım 4 — Hazır Açık Modellerle Aynı Ölçüde Kıyaslama
3 adımdır kendi modelimizi sıfırdan yazdık. Şimdi aynı ölçü çubuğunu (**karakter başına hata**) hazır açık modellere uyguluyoruz:

| Model | Parametre | Eğitim |
|---|---|---|
| Bizim BPE Mini GPT (3. adım) | 3,7 milyon | 4,5 M karakter Türkçe Vikipedi |
| Qwen3-0.6B-Base | 0,6 milyar | 119 dil, çok dilli |
| Qwen3-1.7B-Base | 1,7 milyar | 119 dil, çok dilli |
| Kumru-2B-Base | 2,4 milyar | Sıfırdan Türkçe, ~300 milyar token |

**Cevaplayacağımız sorular:**
1. Ölçek ne kadar fark yaratıyor?
2. Türkçeye özel eğitim, çok dilli bir modele göre ne kazandırıyor? (Kumru hem Türkçe hem daha büyük olduğu için, boyut etkisini ayırmak üzere Kumru'ya yakın boyutta Qwen3-1.7B de eklendi.)
3. Tokenizer'lar Türkçeyi ne kadar verimli bölüyor?

**İki uyarı (sonuçları yorumlarken unutma):**
- **Veri sızıntısı:** Büyük modeller ön eğitimde muhtemelen Türkçe Vikipedi'yi gördü. Vikipedi doğrulama metnimizdeki skorları kısmen ezberden gelebilir. Bu yüzden 7. bölümde **modellerin hiç görmediği** bir metinle de test ediyoruz.
- **Eşit olmayan bağlam:** Bizim modelimiz en fazla 128 token geriye bakabiliyor, büyük modellere 1.024 token veriyoruz.

**Başlamadan önce:** `Çalışma zamanı → L4 GPU` önerilir (T4 de yeterli). İndirme boyutu toplam ~9 GB.

## 1. Kurulum

In [ ]:
%pip install -q "transformers>=4.51" accelerate

In [ ]:
import os, re, gc, math, time, json, unicodedata
from dataclasses import dataclass
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337)
print("Cihaz:", device, "| PyTorch:", torch.__version__)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJE = "/content/drive/MyDrive/yapay_zeka_projesi"
except ImportError:
    PROJE = "./yapay_zeka_projesi"

VERI_DOSYASI = f"{PROJE}/veri/tr_vikipedi.txt"
TOKENIZER_DOSYASI = f"{PROJE}/tokenizer/bpe_tr_2048.json"
CKPT_EN_IYI = f"{PROJE}/checkpoints/mini_gpt_bpe_en_iyi.pt"
for yol in [VERI_DOSYASI, TOKENIZER_DOSYASI, CKPT_EN_IYI]:
    print(("✓ " if os.path.exists(yol) else "✗ EKSİK: ") + yol)

## 2. Bizim modeli ve doğrulama metnini geri yükleme
3. adımdaki tokenizer'ı ve en iyi modeli Drive'dan okuyoruz. Doğrulama metnini de 3. adımdaki adımları **birebir tekrarlayarak** yeniden oluşturuyoruz, böylece tüm modeller tam olarak aynı metinle sınanıyor.

In [ ]:
DONUSUM = str.maketrans({"“": '"', "”": '"', "„": '"', "«": '"', "»": '"',
                         "‘": "'", "’": "'", "–": "-", "—": "-"})
def normallestir(s):
    return unicodedata.normalize("NFC", s).translate(DONUSUM)

ON_AYIRICI = re.compile(r" ?[^\W\d_]+| ?\d{1,3}| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+")

class BPETokenizer:
    def __init__(self, temel, birlesmeler):
        self.temel = list(temel)
        self.birlesmeler = [tuple(c) for c in birlesmeler]
        self.stoi = {c: i for i, c in enumerate(self.temel)}
        self.sira = {c: i for i, c in enumerate(self.birlesmeler)}
        self.sozluk = {i: c for i, c in enumerate(self.temel)}
        for i, (a, b) in enumerate(self.birlesmeler):
            self.sozluk[len(self.temel) + i] = self.sozluk[a] + self.sozluk[b]
        self.onbellek = {}
    def __len__(self):
        return len(self.sozluk)
    def _parca_kodla(self, p):
        if p in self.onbellek:
            return self.onbellek[p]
        k = [self.stoi[c] for c in p if c in self.stoi]
        while len(k) >= 2:
            en_iyi = min(zip(k, k[1:]), key=lambda c: self.sira.get(c, float("inf")))
            if en_iyi not in self.sira:
                break
            yeni_id, yeni_k, i = len(self.temel) + self.sira[en_iyi], [], 0
            while i < len(k):
                if i < len(k) - 1 and (k[i], k[i + 1]) == en_iyi:
                    yeni_k.append(yeni_id); i += 2
                else:
                    yeni_k.append(k[i]); i += 1
            k = yeni_k
        self.onbellek[p] = k
        return k
    def encode(self, s):
        ids = []
        for p in ON_AYIRICI.findall(normallestir(s)):
            ids.extend(self._parca_kodla(p))
        return ids
    def decode(self, ids):
        return "".join(self.sozluk[i] for i in ids)
    @classmethod
    def yukle(cls, yol):
        with open(yol, encoding="utf-8") as f:
            d = json.load(f)
        return cls(d["temel"], d["birlesmeler"])

tok = BPETokenizer.yukle(TOKENIZER_DOSYASI)

with open(VERI_DOSYASI, encoding="utf-8") as f:
    metin = normallestir(f.read())
izinli = set(c for c, n in Counter(metin).items() if n >= 50)
metin = "".join(c for c in metin if c in izinli)

ids = tok.encode(metin)
n = int(0.9 * len(ids))
dogrulama_metni = tok.decode(ids[n:])

# Doğrulama metninin TAMAMI kullanılıyor: küçük bir parça (ör. ilk 100 bin karakter)
# sonucu 0,08'e kadar kaydırabiliyor, çünkü makalelerin zorluğu farklı.
vikipedi_test = dogrulama_metni

# Büyük metin ve token listesi artık gerekmiyor: RAM'i büyük modeller için boşalt
del metin, ids
gc.collect()

print(f"Sözlük: {len(tok)} | Test metni (doğrulama metninin tamamı): {len(vikipedi_test):,} karakter")
print("Test metninin başı:\n", vikipedi_test[:300])

In [ ]:
@dataclass
class Ayar:
    sozluk: int
    blok: int = 128
    gomme: int = 256
    kafa: int = 4
    katman: int = 4
    dropout: float = 0.1

class OzDikkat(nn.Module):
    def __init__(self, a):
        super().__init__()
        self.kafa = a.kafa
        self.qkv = nn.Linear(a.gomme, 3 * a.gomme, bias=False)
        self.cikis = nn.Linear(a.gomme, a.gomme)
        self.drop = nn.Dropout(a.dropout)
        self.register_buffer("maske", torch.tril(torch.ones(a.blok, a.blok)).view(1, 1, a.blok, a.blok))
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        hs = C // self.kafa
        q, k, v = [t.view(B, T, self.kafa, hs).transpose(1, 2) for t in (q, k, v)]
        skor = (q @ k.transpose(-2, -1)) / math.sqrt(hs)
        skor = skor.masked_fill(self.maske[:, :, :T, :T] == 0, float("-inf"))
        y = (self.drop(F.softmax(skor, dim=-1)) @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.cikis(y))

class IleriBesleme(nn.Module):
    def __init__(self, a):
        super().__init__()
        self.ag = nn.Sequential(nn.Linear(a.gomme, 4 * a.gomme), nn.GELU(),
                                nn.Linear(4 * a.gomme, a.gomme), nn.Dropout(a.dropout))
    def forward(self, x):
        return self.ag(x)

class Blok(nn.Module):
    def __init__(self, a):
        super().__init__()
        self.ln1, self.dikkat = nn.LayerNorm(a.gomme), OzDikkat(a)
        self.ln2, self.mlp = nn.LayerNorm(a.gomme), IleriBesleme(a)
    def forward(self, x):
        x = x + self.dikkat(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self, a):
        super().__init__()
        self.a = a
        self.token_gomme = nn.Embedding(a.sozluk, a.gomme)
        self.konum_gomme = nn.Embedding(a.blok, a.gomme)
        self.drop = nn.Dropout(a.dropout)
        self.bloklar = nn.Sequential(*[Blok(a) for _ in range(a.katman)])
        self.ln_son = nn.LayerNorm(a.gomme)
        self.kafa = nn.Linear(a.gomme, a.sozluk, bias=False)
        self.kafa.weight = self.token_gomme.weight
    def forward(self, idx):
        T = idx.shape[1]
        x = self.drop(self.token_gomme(idx) + self.konum_gomme(torch.arange(T, device=idx.device)))
        return self.kafa(self.ln_son(self.bloklar(x)))

kayit = torch.load(CKPT_EN_IYI, map_location=device)
bizim_model = MiniGPT(Ayar(**kayit["ayar"])).to(device).eval()
bizim_model.load_state_dict(kayit["model"])
print(f"Bizim model yüklendi | adım {kayit['adim']} | parametre: {sum(p.numel() for p in bizim_model.parameters()):,}")
print(f"3. adımdaki karakter başına doğrulama hatası: {kayit['dogrulama_kar']:.3f}")

## 3. Ortak ölçü: karakter başına hata
Her model metni kendi tokenizer'ıyla böler. Metni pencerelere ayırıp her tokenin olasılığını ölçüyoruz; toplam hatayı, **tahmin edilen tokenların kapsadığı karakter sayısına** bölüyoruz.
Böylece 2.048, 50 bin ya da 150 bin tokenlık sözlükler aynı ölçekte karşılaştırılabiliyor.

In [ ]:
@torch.no_grad()
def bizim_kar_basi(metin):
    ids = tok.encode(metin)
    blok = bizim_model.a.blok
    toplam_kayip, toplam_kar = 0.0, 0
    for s in range(0, len(ids) - 1, blok):
        parca = ids[s:s + blok + 1]
        if len(parca) < 2:
            break
        x = torch.tensor([parca[:-1]], device=device)
        y = torch.tensor([parca[1:]], device=device)
        logits = bizim_model(x)
        toplam_kayip += F.cross_entropy(logits[0].float(), y[0], reduction="sum").item()
        toplam_kar += len(tok.decode(parca[1:]))
    return toplam_kayip / toplam_kar

t0 = time.time()
sonuclar = {}
sonuclar["Bizim BPE Mini GPT"] = {"parametre": sum(p.numel() for p in bizim_model.parameters()),
                                  "vikipedi": bizim_kar_basi(vikipedi_test),
                                  "kar_tok": len(vikipedi_test) / len(tok.encode(vikipedi_test))}
print(f"Bizim model: {sonuclar['Bizim BPE Mini GPT']['vikipedi']:.3f} karakter başına ({time.time() - t0:.0f} sn)")
print("(3. adımdaki değerle neredeyse aynı çıkmalı; küçük fark, burada sabit pencereler kullanmamızdan.)")

## 4. Hugging Face modelleri
Modelleri sırayla yükleyip ölçüyor, sonra bellekten siliyoruz; iki model aynı anda T4'e sığmaz.
T4 `bfloat16`'yı verimli desteklemediği için ve `float16`'nın bu modellerde taşma riski olduğu için **float32** kullanıyoruz (2B model ≈ 8 GB VRAM).

**Bellek notu:** Colab'ın ücretsiz sürümünde ~12,7 GB RAM var. Bir modeli önce RAM'de float32'ye çevirip sonra GPU'ya taşımak, 2B modelde RAM'i taşırıp oturumu çökertir. `device_map` ile ağırlıklar parça parça **doğrudan GPU'ya** yüklenir; RAM'de tüm modelin aynı anda durması gerekmez.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELLER = {
    "Qwen3-0.6B-Base": "Qwen/Qwen3-0.6B-Base",
    "Kumru-2B-Base": "vngrs-ai/Kumru-2B-Base",
    "Qwen3-1.7B-Base": "Qwen/Qwen3-1.7B-Base",
}
HF_BAGLAM = 1024

def hf_yukle(model_id):
    hf_tok = AutoTokenizer.from_pretrained(model_id)
    hf_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32,
                                                    low_cpu_mem_usage=True, device_map=device).eval()
    return hf_tok, hf_model

@torch.no_grad()
def hf_kar_basi(hf_tok, hf_model, metin, baglam=HF_BAGLAM):
    kod = hf_tok(metin, return_offsets_mapping=True, add_special_tokens=False)
    ids, ofs = kod["input_ids"], kod["offset_mapping"]
    # Bazı modeller (ör. Kumru) her girdinin başında <BOS> bekler; olmadan tamamen bozulur
    # (Kumru'da BOS'suz token başına hata 15,9, BOS'lu 1,5). BOS kullanmayan modellerde liste boş kalır.
    bos = [hf_tok.bos_token_id] if hf_tok.bos_token_id is not None else []
    toplam_kayip, toplam_kar = 0.0, 0
    for s in range(0, len(ids) - 1, baglam):
        e = min(s + baglam + 1, len(ids))
        if e - s < 2:
            break
        x = torch.tensor([bos + ids[s:e - 1]], device=device)
        y = torch.tensor([ids[s + 1:e]], device=device)
        logits = hf_model(x).logits[0, len(bos):].float()      # BOS'un kendi tahminini atla
        toplam_kayip += F.cross_entropy(logits, y[0], reduction="sum").item()
        toplam_kar += ofs[e - 1][1] - ofs[s][1]      # tahmin edilen tokenların kapsadığı karakterler
    return toplam_kayip / toplam_kar

def bellegi_bosalt():
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

In [ ]:
for ad, model_id in MODELLER.items():
    print(f"=== {ad} ({model_id}) ===")
    t0 = time.time()
    try:
        hf_tok, hf_model = hf_yukle(model_id)
    except Exception as hata:
        print(f"  Yüklenemedi: {type(hata).__name__}: {hata}\n")
        continue
    p = sum(x.numel() for x in hf_model.parameters())
    kayip = hf_kar_basi(hf_tok, hf_model, vikipedi_test)
    if math.isnan(kayip):
        print("  Uyarı: sonuç NaN çıktı (sayısal taşma).")
    sonuclar[ad] = {"parametre": p, "vikipedi": kayip,
                    "kar_tok": len(vikipedi_test) / len(hf_tok(vikipedi_test, add_special_tokens=False)["input_ids"]),
                    "sozluk": len(hf_tok)}
    print(f"  Parametre: {p / 1e9:.2f} B | sözlük: {len(hf_tok):,} | "
          f"karakter/token: {sonuclar[ad]['kar_tok']:.2f} | karakter başına hata: {kayip:.3f} | {time.time() - t0:.0f} sn\n")
    del hf_model, hf_tok
    bellegi_bosalt()

In [ ]:
print(f"{'Model':<22} {'Parametre':>12} {'Kar/token':>10} {'Vikipedi (kar. başına)':>24}")
print("-" * 72)
for ad, s in sonuclar.items():
    p = s["parametre"]
    p_str = f"{p / 1e9:.2f} B" if p >= 1e8 else f"{p / 1e6:.1f} M"
    print(f"{ad:<22} {p_str:>12} {s['kar_tok']:>10.2f} {s['vikipedi']:>24.3f}")
print("\nReferans: 2. adım karakter modeli 1.275")

### BOS teşhisi: bir ölçüm hatası nasıl bulundu?
İlk ölçümde Kumru-2B **2,277** çıktı: 2 milyar parametrelik, Türkçe için eğitilmiş bir modelin 3,7 milyonluk modelimizden kötü olması mümkün değildi.
Sebebi: Kumru her girdinin başında özel bir `<BOS>` (başlangıç) tokenı bekliyor. Pencereler BOS'suz verildiğinde model tamamen bozuluyor.
Ölçüm fonksiyonu bu yüzden BOS bekleyen modellere her pencerenin başına BOS ekliyor. Aşağıdaki hücre farkı gösteriyor.

In [ ]:
hf_tok, hf_model = hf_yukle("vngrs-ai/Kumru-2B-Base")
print("Mimari:", type(hf_model).__name__, "| BOS:", hf_tok.bos_token, "| EOS:", hf_tok.eos_token)

ornek = "Türkiye'nin başkenti Ankara'dır. Ankara, İç Anadolu Bölgesi'nde yer alır."
print("Tokenlar:", [hf_tok.decode([i]) for i in hf_tok(ornek, add_special_tokens=False)["input_ids"]])
for bos in [False, True]:
    ids = hf_tok(ornek, add_special_tokens=bos, return_tensors="pt")["input_ids"].to(device)
    with torch.no_grad():
        print(("BOS'lu " if bos else "BOS'suz"), "token başına hata:", round(hf_model(ids, labels=ids).loss.item(), 3))

del hf_model, hf_tok
bellegi_bosalt()

## 5. Tokenizer karşılaştırması
Aynı cümleyi üç tokenizer nasıl bölüyor? Türkçeye özel eğitilmiş bir tokenizer ekleri ve kökleri daha doğal ayırmalı.

In [ ]:
cumleler = ["Öğrencilerimizin başarısı bizi gururlandırıyor.",
            "Mesleki ve teknik anadolu lisesi öğretmenleri toplantıya katılacaktır.",
            "Osmanlı İmparatorluğu 1299 yılında kuruldu."]

hf_tokenizerlar = {}
for ad, model_id in MODELLER.items():
    try:
        hf_tokenizerlar[ad] = AutoTokenizer.from_pretrained(model_id)
    except Exception as hata:
        print(f"{ad} tokenizer'ı yüklenemedi: {hata}")

for c in cumleler:
    print(f"\n{c!r}")
    bizim = [tok.sozluk[i] for i in tok.encode(c)]
    print(f"  Bizim BPE ({len(bizim):2d} token): {bizim}")
    for ad, t in hf_tokenizerlar.items():
        parcalar = [t.decode([i]) for i in t(c, add_special_tokens=False)["input_ids"]]
        print(f"  {ad:<17}({len(parcalar):2d} token): {parcalar}")

## 6. Metin üretimi
Base modeller sohbet etmez; verilen metni **devam ettirir**. Aynı başlangıçla üç modelin yazdığına bakalım.

In [ ]:
@torch.no_grad()
def bizim_uret(baslangic, uzunluk=80, sicaklik=0.8, top_k=40):
    idx = torch.tensor([tok.encode(baslangic)], device=device)
    for _ in range(uzunluk):
        logits = bizim_model(idx[:, -bizim_model.a.blok:])[:, -1, :] / sicaklik
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float("inf")
        idx = torch.cat([idx, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
    return tok.decode(idx[0].tolist())

BASLANGIC = "Türkiye'nin başkenti Ankara,"

print("=== Bizim BPE Mini GPT ===")
print(bizim_uret(BASLANGIC), "\n")

for ad, model_id in MODELLER.items():
    try:
        hf_tok, hf_model = hf_yukle(model_id)
    except Exception as hata:
        print(f"=== {ad}: yüklenemedi ({hata}) ===\n")
        continue
    girdi = hf_tok(BASLANGIC, return_tensors="pt").to(device)
    torch.manual_seed(1337)
    cikti = hf_model.generate(**girdi, max_new_tokens=60, do_sample=True, temperature=0.8, top_k=40,
                              pad_token_id=hf_tok.eos_token_id)
    print(f"=== {ad} ===")
    print(hf_tok.decode(cikti[0], skip_special_tokens=True), "\n")
    del hf_model, hf_tok
    bellegi_bosalt()

## 7. Sızıntısız test: modellerin hiç görmediği metin
Vikipedi skorları büyük modelleri **olduğundan iyi** gösterebilir, çünkü bu metinleri ön eğitimde görmüş olabilirler.
Gerçek genelleme yeteneğini görmek için aşağıya **kendi yazdığın** bir metni yapıştır: bir okul duyurusu, veli mektubu ya da toplantı tutanağı. En az birkaç paragraf olsun (ideal olarak 3.000+ karakter).

Vikipedi skoru ile bu skor arasındaki fark ne kadar büyükse, Vikipedi skorunun o kadarı "ezber" olabilir.

In [ ]:
KENDI_METNIM = """
Buraya kendi yazdığın bir metni yapıştır. Örneğin bir okul duyurusu, veli bilgilendirme yazısı
ya da toplantı tutanağı. Metin internette yayımlanmamış olmalı ki modeller onu daha önce görmemiş olsun.
""".strip()

print(f"Metin uzunluğu: {len(KENDI_METNIM):,} karakter")
if len(KENDI_METNIM) < 1000:
    print("Uyarı: Metin kısa; sonuçlar gürültülü olabilir. 3.000+ karakter önerilir.")

sonuclar["Bizim BPE Mini GPT"]["kendi"] = bizim_kar_basi(KENDI_METNIM)
for ad, model_id in MODELLER.items():
    if ad not in sonuclar:
        continue
    hf_tok, hf_model = hf_yukle(model_id)
    sonuclar[ad]["kendi"] = hf_kar_basi(hf_tok, hf_model, KENDI_METNIM)
    del hf_model, hf_tok
    bellegi_bosalt()

print(f"\n{'Model':<22} {'Vikipedi':>10} {'Kendi metnim':>14} {'Fark':>8}")
print("-" * 58)
for ad, s in sonuclar.items():
    if "kendi" in s:
        print(f"{ad:<22} {s['vikipedi']:>10.3f} {s['kendi']:>14.3f} {s['kendi'] - s['vikipedi']:>+8.3f}")

## 8. Alıştırmalar
1. **Bağlamı eşitle:** `hf_kar_basi(..., baglam=128)` ile büyük modelleri de kısa bağlamla ölç. Farkın ne kadarı bağlamdan geliyordu?
2. **Talimatlı model:** `vngrs-ai/Kumru-2B` (Base olmayan) ile aynı ölçümü yap. Talimat eğitimi karakter başına hatayı nasıl etkiliyor?
3. **Birden fazla metin:** 7. bölümü farklı türde birkaç metinle (duyuru, tutanak, yönetmelik) tekrarla. Tek metindeki küçük farklar gürültü olabilir.

## Sonuçlar (L4 GPU, float32)
| Model | Parametre | Karakter/token | Vikipedi | Kendi metnim | Fark |
|---|---|---|---|---|---|
| Bizim BPE Mini GPT | 3,7 M | 2,62 | 1,128 | 1,428 | +0,300 |
| Qwen3-0.6B-Base | 0,60 B | 2,45 | 0,965 | 0,923 | −0,042 |
| Qwen3-1.7B-Base | 1,72 B | 2,45 | 0,780 | 0,828 | +0,048 |
| Kumru-2B-Base | 2,38 B | 3,11 | 0,611 | 0,688 | +0,077 |

**Özet:** Büyük modeller hiç görmedikleri metinde de Vikipedi'deki seviyelerini korudu; skorları ezber değil, genelleme. Bizim model ise dar eğitim alanı yüzünden %27 kötüleşti. Kumru'nun Qwen-1.7B'ye üstünlüğü Vikipedi dışında da sürdü; bu üstünlüğün kabaca üçte ikisi Türkçeye özel eğitimden, üçte biri boyuttan geliyor gibi görünüyor.